In [ ]:
# downloading the data from the Polymarket API and saving it to a CSV file

import asyncio
import aiohttp
import pandas as pd
from tqdm.notebook import tqdm
import os

BASE_URL = "https://gamma-api.polymarket.com/markets/keyset"

TARGET_ROWS = 200_000
LIMIT = 500

all_markets = []

CSV_PATH = "Polymarket - workspace/markets_data.csv"


async def fetch_page(session, cursor=None):

    params = {
        "limit": LIMIT,
        "closed": "true"
    }

    if cursor:
        params["after_cursor"] = cursor

    for attempt in range(5):

        try:

            async with session.get(BASE_URL, params=params) as response:

                if response.status != 200:
                    text = await response.text()
                    print("STATUS:", response.status)
                    print(text)

                response.raise_for_status()

                return await response.json()

        except Exception as e:

            print(f"Fehler: {e} | Versuch {attempt+1}/5")

            await asyncio.sleep(2)

    return None


async def main():

    global all_markets

    timeout = aiohttp.ClientTimeout(total=120)

    connector = aiohttp.TCPConnector(limit=10)

    async with aiohttp.ClientSession(
        timeout=timeout,
        connector=connector
    ) as session:

        cursor = None

        with tqdm(total=TARGET_ROWS) as pbar:

            while len(all_markets) < TARGET_ROWS:

                data = await fetch_page(session, cursor)

                if data is None:
                    print("Abbruch wegen mehrfacher Fehler")
                    break

                markets = data.get("markets", [])

                if len(markets) == 0:
                    print("Keine weiteren Markets")
                    break

                all_markets.extend(markets)

                pbar.update(len(markets))

                cursor = data.get("next_cursor")

                if not cursor or cursor == "LTE=":
                    print("Ende erreicht")
                    break

    return pd.DataFrame(all_markets[:TARGET_ROWS])


if os.path.exists(CSV_PATH):
    df_markets = pd.read_csv(CSV_PATH)
    print(f"Loaded from file: {df_markets.shape}")

else:
    df_markets = await main()
    df_markets.to_csv(CSV_PATH, index=False)
    print(f"Fetched from API and saved: {df_markets.shape}")

print(df_markets.shape)